# ARC-AGI Solver — Combined Training: RE-ARC + Extra Synthetic Data

Trains the decoder-only transformer on a combined pool of synthetic data:
- **RE-ARC**: 1,000 generated examples per task (400 tasks)
- **arc_extra**: up to 262 additional examples per task (400 tasks, same format)

All examples are pooled and sampled randomly — **no leave-one-out during training**.
The shuffled pool gives ~1,262 examples per task; the training window samples
800 of these each epoch, ensuring the extra examples reach the model.

**Stratified split** (from `data/task_split.json`, category-aware):
- **Train**: 278 tasks — combined synthetic pool for training
- **Val**: 81 tasks — ARC leave-one-out exact match, used for checkpoint selection
- **Eval**: 41 tasks — held out completely, never seen during training or selection

**Before running Cell 4:** upload `Archive.zip` to your Google Drive under
`MyDrive/arc-agi/arc_extra.zip`.

Checkpoint name: `transformer_ccombined_278_best.pt`

**Cell order:** 1 → 2 → 3 → 4 → 5 → 6

In [ ]:
# ── Cell 1: Check GPU ───────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — training on CPU will be extremely slow.')

In [ ]:
# ── Cell 2: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_BASE     = '/content/drive/MyDrive/arc-agi'
DRIVE_CKPT_DIR = f'{DRIVE_BASE}/checkpoints'

Path(DRIVE_CKPT_DIR).mkdir(parents=True, exist_ok=True)

print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')

In [ ]:
# ── Cell 3: Clone repo and download ARC training data ───────────────────────
import os, io, subprocess, urllib.request, zipfile
from pathlib import Path

REPO        = 'arc-agi-solver'
GITHUB_USER = 'rodehyde'
REPO_DIR    = f'/content/{REPO}'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', f'https://github.com/{GITHUB_USER}/{REPO}.git', REPO_DIR],
                   check=True)
else:
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'],
                            capture_output=True, text=True)
    print(result.stdout or result.stderr)

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
log = subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-3'],
                     capture_output=True, text=True)
print(log.stdout)

# Download ARC training data
DATA_DIR  = Path('data')
train_dir = DATA_DIR / 'training'
if not train_dir.exists() or len(list(train_dir.glob('*.json'))) < 400:
    print('Downloading ARC training data...')
    with urllib.request.urlopen(
        'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip'
    ) as r:
        raw = r.read()
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        dest = DATA_DIR / 'training'
        dest.mkdir(exist_ok=True)
        for m in zf.namelist():
            if 'data/training/' in m and m.endswith('.json'):
                (dest / Path(m).name).write_bytes(zf.read(m))
    print(f"training: {len(list(train_dir.glob('*.json')))} tasks")
else:
    print(f"ARC training data already present ({len(list(train_dir.glob('*.json')))} tasks)")

In [ ]:
# ── Cell 4: Download RE-ARC and extract arc_extra ───────────────────────────
# RE-ARC: 400 tasks × 1,000 examples each → data/re_arc/
# arc_extra: Upload Archive.zip to Drive as arc-agi/arc_extra.zip first.
import subprocess, sys, zipfile, shutil
from pathlib import Path

# ── RE-ARC ──────────────────────────────────────────────────────────────────
RE_ARC_DIR = Path('data/re_arc')
if RE_ARC_DIR.exists() and len(list(RE_ARC_DIR.glob('*.json'))) >= 400:
    print(f'RE-ARC already present ({len(list(RE_ARC_DIR.glob("*.json")))} tasks)')
else:
    print('Downloading RE-ARC synthetic data...')
    result = subprocess.run([sys.executable, 'scripts/download_re_arc.py'],
                            capture_output=True, text=True)
    out = result.stdout
    print(out[-3000:] if len(out) > 3000 else out)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
    print(f'RE-ARC: {len(list(RE_ARC_DIR.glob("*.json")))} tasks')

# ── arc_extra ───────────────────────────────────────────────────────────────
EXTRA_DIR      = Path('data/arc_extra')
EXTRA_ZIP_DRIVE = Path(DRIVE_BASE) / 'arc_extra.zip'

if EXTRA_DIR.exists() and len(list(EXTRA_DIR.glob('*.json'))) >= 400:
    print(f'arc_extra already present ({len(list(EXTRA_DIR.glob("*.json")))} tasks)')
elif not EXTRA_ZIP_DRIVE.exists():
    print(f'WARNING: {EXTRA_ZIP_DRIVE} not found on Drive.')
    print('Upload Archive.zip to your Google Drive arc-agi folder as arc_extra.zip')
    print('Then re-run this cell.  Training will continue with RE-ARC only until then.')
else:
    print(f'Extracting arc_extra from Drive ({EXTRA_ZIP_DRIVE.stat().st_size / 1e6:.1f} MB)...')
    EXTRA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(EXTRA_ZIP_DRIVE) as zf:
        zf.extractall(EXTRA_DIR)
    print(f'arc_extra: {len(list(EXTRA_DIR.glob("*.json")))} tasks')

In [ ]:
# ── Cell 5: Configuration ────────────────────────────────────────────────────
import json
from pathlib import Path

RUN_NAME        = 'combined_278'
K_CONTEXT       = 3
D_MODEL         = 512
N_HEADS         = 8
N_LAYERS        = 6
LR              = 3e-4
WARMUP_EPOCHS   = 5
EPOCHS          = 300
STEPS_PER_EPOCH = 400
MAX_TOKENS      = 96000
SAVE_EVERY      = 25
LOG_EVERY       = 1
LOG_FILE        = f'results/train_{RUN_NAME}.log'

split        = json.loads(Path('data/task_split.json').read_text())
TASK_IDS     = split['train']   # 278 tasks
VAL_TASK_IDS = split['val']     # 81 tasks — checkpoint selection

extra_dir  = Path('data/arc_extra')
n_extra    = sum(1 for tid in TASK_IDS if (extra_dir / f'{tid}.json').exists()) if extra_dir.exists() else 0
data_source = 'combined' if n_extra > 0 else 'rearc'

print(f'Run:         {RUN_NAME}')
print(f'Data source: {data_source}')
print(f'Train tasks: {len(TASK_IDS)}  ({n_extra} have arc_extra data)')
print(f'Val tasks:   {len(VAL_TASK_IDS)}  (ARC LOO exact match — checkpoint criterion)')
print(f'Eval tasks:  {len(split["eval"])}  (held out completely)')
print(f'Epochs:      {EPOCHS} × {STEPS_PER_EPOCH} steps = {EPOCHS*STEPS_PER_EPOCH:,} total')
print(f'd_model:     {D_MODEL},  heads: {N_HEADS},  layers: {N_LAYERS},  k_ctx: {K_CONTEXT}')
print(f'LR:          {LR},  warmup: {WARMUP_EPOCHS} epochs')

In [ ]:
# ── Cell 6: Run training ─────────────────────────────────────────────────────
# Data source: combined (RE-ARC + arc_extra pooled, no LOO during training).
# Falls back to rearc-only if arc_extra was not found in Cell 4.
# Checkpoint criterion: highest ARC LOO exact match on VAL_TASK_IDS.
import subprocess, sys

cmd = [
    sys.executable, '-u', 'scripts/train_transformer.py',
    '--data-source',      data_source,
    '--task-ids',         *TASK_IDS,
    '--val-arc-task-ids', *VAL_TASK_IDS,
    '--run-name',         RUN_NAME,
    '--epochs',           str(EPOCHS),
    '--steps-per-epoch',  str(STEPS_PER_EPOCH),
    '--k-context',        str(K_CONTEXT),
    '--d-model',          str(D_MODEL),
    '--n-heads',          str(N_HEADS),
    '--n-layers',         str(N_LAYERS),
    '--lr',               str(LR),
    '--warmup-epochs',    str(WARMUP_EPOCHS),
    '--max-tokens',       str(MAX_TOKENS),
    '--max-seq-len',      '6000',
    '--save-every',       str(SAVE_EVERY),
    '--log-every',        str(LOG_EVERY),
    '--log',              LOG_FILE,
    '--ckpt-dir',         DRIVE_CKPT_DIR,
]

print(f'Checkpoints → {DRIVE_CKPT_DIR}')
print(f'Log         → {LOG_FILE}')
print('=' * 60)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'Exit code: {proc.returncode}')

In [ ]:
# ── Cell 7: View training log (last 40 lines) ───────────────────────────────
from pathlib import Path
log = Path(LOG_FILE)
if log.exists():
    lines = log.read_text().splitlines()
    print(f'Log: {log}  ({len(lines)} lines)')
    print('\n'.join(lines[-40:]))
else:
    print('Log file not found — training may not have started yet.')